In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType

spark = (SparkSession.builder
         .master("local[*]")
         .appName("monografia")
         .config("spark.driver.memory", "8g")
         .getOrCreate())

#leer archivo
df = spark.read.parquet("../data/secop.parquet")
print("Contratos:", df.count())

Contratos: 2734292


In [2]:
# Q1: número de contratos y valor total por departamento ----------
# ¿cuántos contratos firmó cada departamento entre 2023 y 2025, y cuánto dinero suman?

# Variante 1: RDD
def q1_rdd(df):
    return (df.select("departamento", "valor").rdd
              .map(lambda r: ((r[0] or "").strip().upper(), (1, r[1])))
              .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))

# Variante 2: DataFrame
def q1_df(df):
    return (df.withColumn("dep", F.upper(F.trim("departamento")))
              .groupBy("dep")
              .agg(F.count("*").alias("n"), F.sum("valor").alias("total")))

# Variante 3: Spark SQL
def q1_sql(df):
    df.createOrReplaceTempView("secop")
    return spark.sql("""
        SELECT upper(trim(departamento)) AS dep,
               count(*)                  AS n,
               sum(valor)                AS total
        FROM secop
        GROUP BY upper(trim(departamento))
    """)

# Variante 4: DataFrame + UDF
normalizar = F.udf(lambda s: (s or "").strip().upper(), StringType())

def q1_udf(df):
    return (df.withColumn("dep", normalizar("departamento"))
              .groupBy("dep")
              .agg(F.count("*").alias("n"), F.sum("valor").alias("total")))

In [3]:
print("RDD:")
for fila in q1_rdd(df).takeOrdered(5, key=lambda x: x[0]):
    print(fila)

print("\nDataFrame:")
q1_df(df).orderBy("dep").show(5, truncate=False)

print("SQL:")
q1_sql(df).orderBy("dep").show(5, truncate=False)

print("UDF:")
q1_udf(df).orderBy("dep").show(5, truncate=False)

RDD:
('AMAZONAS', (9143, 770822278614.5602))
('ANTIOQUIA', (278581, 199730051624170.1))
('ARAUCA', (10562, 1793540851207.4502))
('ATLÁNTICO', (91927, 14322046607146.43))
('BOLÍVAR', (83417, 13055551790005.559))

DataFrame:
+---------+------+---------------------+
|dep      |n     |total                |
+---------+------+---------------------+
|AMAZONAS |9143  |7.708222786145602E11 |
|ANTIOQUIA|278581|1.997300516241701E14 |
|ARAUCA   |10562 |1.7935408512074502E12|
|ATLÁNTICO|91927 |1.432204660714643E13 |
|BOLÍVAR  |83417 |1.3055551790005559E13|
+---------+------+---------------------+
only showing top 5 rows

SQL:
+---------+------+---------------------+
|dep      |n     |total                |
+---------+------+---------------------+
|AMAZONAS |9143  |7.708222786145602E11 |
|ANTIOQUIA|278581|1.997300516241701E14 |
|ARAUCA   |10562 |1.7935408512074502E12|
|ATLÁNTICO|91927 |1.432204660714643E13 |
|BOLÍVAR  |83417 |1.3055551790005559E13|
+---------+------+---------------------+
only show

In [4]:
# Q2: contratos grandes de Antioquia
#  ¿qué contratos de Antioquia superan los 100 millones de pesos?

UMBRAL = 100000000          # 100 millones de pesos

# Variante 1: RDD
def q2_rdd(df):
    return (df.select("id_contrato", "nombre_entidad", "fecha", "valor", "departamento").rdd
              .filter(lambda r: (r[4] or "").strip().upper() == "ANTIOQUIA" and r[3] > UMBRAL)
              .map(lambda r: (r[0], r[1], r[2], r[3])))

# Variante 2: DataFrame
def q2_df(df):
    return (df.filter((F.upper(F.trim("departamento")) == "ANTIOQUIA") & (F.col("valor") > UMBRAL))
              .select("id_contrato", "nombre_entidad", "fecha", "valor"))

# Variante 3: Spark SQL
def q2_sql(df):
    df.createOrReplaceTempView("secop")
    return spark.sql(f"""
        SELECT id_contrato, nombre_entidad, fecha, valor
        FROM secop
        WHERE upper(trim(departamento)) = 'ANTIOQUIA'
          AND valor > {UMBRAL}
    """)

# Variante 4: DataFrame + UDF
def q2_udf(df):
    return (df.filter((normalizar("departamento") == "ANTIOQUIA") & (F.col("valor") > UMBRAL))
              .select("id_contrato", "nombre_entidad", "fecha", "valor"))

In [5]:
print("Cantidad de contratos que cumplen el filtro:")
print("  RDD:      ", q2_rdd(df).count())
print("  DataFrame:", q2_df(df).count())
print("  SQL:      ", q2_sql(df).count())
print("  UDF:      ", q2_udf(df).count())

print("\nPrimeros 3 por id_contrato (RDD):")
for fila in q2_rdd(df).takeOrdered(3, key=lambda x: x[0]):
    print(fila)

print("\nPrimeros 3 por id_contrato (DataFrame):")
q2_df(df).orderBy("id_contrato").show(3, truncate=False)

Cantidad de contratos que cumplen el filtro:
  RDD:       25514
  DataFrame: 25514
  SQL:       25514
  UDF:       25514

Primeros 3 por id_contrato (RDD):
('CO1.PCCNTR.1698725', 'DEPARTAMENTO DE ANTIOQUIA//', datetime.date(2023, 6, 2), 198526320.0)
('CO1.PCCNTR.1699658', 'DEPARTAMENTO DE ANTIOQUIA//', datetime.date(2023, 7, 14), 436697200.0)
('CO1.PCCNTR.1827739', 'DEPARTAMENTO DE ANTIOQUIA//', datetime.date(2023, 7, 27), 114453500.0)

Primeros 3 por id_contrato (DataFrame):
+------------------+---------------------------+----------+-----------+
|id_contrato       |nombre_entidad             |fecha     |valor      |
+------------------+---------------------------+----------+-----------+
|CO1.PCCNTR.1698725|DEPARTAMENTO DE ANTIOQUIA//|2023-06-02|1.9852632E8|
|CO1.PCCNTR.1699658|DEPARTAMENTO DE ANTIOQUIA//|2023-07-14|4.366972E8 |
|CO1.PCCNTR.1827739|DEPARTAMENTO DE ANTIOQUIA//|2023-07-27|1.144535E8 |
+------------------+---------------------------+----------+-----------+
only showing to

In [6]:
# Q3: valor total por modalidad y mes 
# ¿cuánto dinero se contrató cada mes con cada modalidad de contratación ?

# Variante 1: RDD
def q3_rdd(df):
    return (df.select("modalidad_de_contratacion", "fecha", "valor").rdd
              .map(lambda r: (((r[0] or "").strip().upper(), r[1].strftime("%Y-%m")), r[2]))
              .reduceByKey(lambda a, b: a + b))

# Variante 2: DataFrame
def q3_df(df):
    return (df.withColumn("modalidad", F.upper(F.trim("modalidad_de_contratacion")))
              .withColumn("mes", F.date_format("fecha", "yyyy-MM"))
              .groupBy("modalidad", "mes")
              .agg(F.sum("valor").alias("total")))

# Variante 3: Spark SQL
def q3_sql(df):
    df.createOrReplaceTempView("secop")
    return spark.sql("""
        SELECT upper(trim(modalidad_de_contratacion)) AS modalidad,
               date_format(fecha, 'yyyy-MM')          AS mes,
               sum(valor)                             AS total
        FROM secop
        GROUP BY upper(trim(modalidad_de_contratacion)),
                 date_format(fecha, 'yyyy-MM')
    """)

# Variante 4: DataFrame + UDF
def q3_udf(df):
    return (df.withColumn("modalidad", normalizar("modalidad_de_contratacion"))
              .withColumn("mes", F.date_format("fecha", "yyyy-MM"))
              .groupBy("modalidad", "mes")
              .agg(F.sum("valor").alias("total")))

In [7]:
# COMPARACIÓN Q3

print("Número de grupos (modalidad, mes):")
print("  RDD:      ", q3_rdd(df).count())
print("  DataFrame:", q3_df(df).count())
print("  SQL:      ", q3_sql(df).count())
print("  UDF:      ", q3_udf(df).count())

print("\nPrimeros 3 (RDD):")
for fila in q3_rdd(df).takeOrdered(3, key=lambda x: x[0]):
    print(fila)

print("\nPrimeros 3 (DataFrame):")
q3_df(df).orderBy("modalidad", "mes").show(3, truncate=False)

Número de grupos (modalidad, mes):
  RDD:       465
  DataFrame: 465
  SQL:       465
  UDF:       465

Primeros 3 (RDD):
(('CONCURSO DE MÉRITOS ABIERTO', '2023-01'), 37096126824.83)
(('CONCURSO DE MÉRITOS ABIERTO', '2023-02'), 67889795380.959984)
(('CONCURSO DE MÉRITOS ABIERTO', '2023-03'), 135813975307.51999)

Primeros 3 (DataFrame):
+---------------------------+-------+---------------------+
|modalidad                  |mes    |total                |
+---------------------------+-------+---------------------+
|CONCURSO DE MÉRITOS ABIERTO|2023-01|3.709612682483E10    |
|CONCURSO DE MÉRITOS ABIERTO|2023-02|6.7889795380959984E10|
|CONCURSO DE MÉRITOS ABIERTO|2023-03|1.3581397530751999E11|
+---------------------------+-------+---------------------+
only showing top 3 rows



In [8]:
#Q4: ranking de proveedores por valor adjudicado
#¿qué proveedores recibieron más dinero en contratos entre 2023 y 2025?

# Variante 1: RDD
def q4_rdd(df):
    return (df.select("proveedor_adjudicado", "valor").rdd
              .map(lambda r: ((r[0] or "").strip().upper(), r[1]))
              .reduceByKey(lambda a, b: a + b)
              .sortBy(lambda x: x[1], ascending=False))

# Variante 2: DataFrame
def q4_df(df):
    return (df.withColumn("proveedor", F.upper(F.trim("proveedor_adjudicado")))
              .groupBy("proveedor")
              .agg(F.sum("valor").alias("total"))
              .orderBy(F.desc("total")))

# Variante 3: Spark SQL
def q4_sql(df):
    df.createOrReplaceTempView("secop")
    return spark.sql("""
        SELECT upper(trim(proveedor_adjudicado)) AS proveedor,
               sum(valor)                         AS total
        FROM secop
        GROUP BY upper(trim(proveedor_adjudicado))
        ORDER BY total DESC
    """)

# Variante 4: DataFrame + UDF
def q4_udf(df):
    return (df.withColumn("proveedor", normalizar("proveedor_adjudicado"))
              .groupBy("proveedor")
              .agg(F.sum("valor").alias("total"))
              .orderBy(F.desc("total")))


In [9]:
import hashlib

def codigo(nombre):
    # Convierte el nombre en un código de 8 caracteres; no se puede revertir
    return hashlib.md5((nombre or "").encode("utf-8")).hexdigest()[:8]

print("Número de proveedores distintos:")
print("  RDD:      ", q4_rdd(df).count())
print("  DataFrame:", q4_df(df).count())
print("  SQL:      ", q4_sql(df).count())
print("  UDF:      ", q4_udf(df).count())

print("\nPrimeros 5 (RDD):")
for i, (prov, total) in enumerate(q4_rdd(df).take(5), start=1):
    print(f"  {i}. proveedor {codigo(prov)}  total {total:,.0f}")

print("\nPrimeros 5 (DataFrame):")
for i, fila in enumerate(q4_df(df).take(5), start=1):
    print(f"  {i}. proveedor {codigo(fila['proveedor'])}  total {fila['total']:,.0f}")

Número de proveedores distintos:
  RDD:       870778
  DataFrame: 870778
  SQL:       870778
  UDF:       870778

Primeros 5 (RDD):
  1. proveedor 2c84c41d  total 6,453,840,718,723,964
  2. proveedor fe9bd8f4  total 944,187,482,659,000
  3. proveedor 0a27d801  total 142,077,621,124,695
  4. proveedor 525aeb5f  total 62,879,300,988,006
  5. proveedor b5767efb  total 62,195,338,760,000

Primeros 5 (DataFrame):
  1. proveedor 2c84c41d  total 6,453,840,718,723,964
  2. proveedor fe9bd8f4  total 944,187,482,659,000
  3. proveedor 0a27d801  total 142,077,621,124,695
  4. proveedor 525aeb5f  total 62,879,300,988,006
  5. proveedor b5767efb  total 62,195,338,760,000


In [10]:
col = F.col("proveedor_adjudicado")

# Grupos tal como los cuenta Spark (DataFrame y SQL)
a = df.select(F.upper(F.trim(col)).alias("k")).distinct().count()

# Grupos si los nulos se tratan como texto vacío (como hace Python)
b = df.select(F.upper(F.trim(F.coalesce(col, F.lit("")))).alias("k")).distinct().count()

# Grupos si además se borran espacios especiales (como el strip de Python)
espacios = r"^[\s\u00A0\u2007\u202F\uFEFF]+|[\s\u00A0\u2007\u202F\uFEFF]+$"
c = df.select(F.upper(F.regexp_replace(F.coalesce(col, F.lit("")), espacios, "")).alias("k")).distinct().count()

print("A. Como Spark:                        ", a)
print("B. Nulos tratados como vacío:         ", b)
print("C. B + quitando espacios especiales:  ", c)

A. Como Spark:                         870778
B. Nulos tratados como vacío:          870778
C. B + quitando espacios especiales:   870778
